# Adapter Pattern — Example

Scenario: our app's client code only knows how to work with a `JSONDataSource` interface (a `fetch_json() -> dict` method).

We are handed a legacy/third-party class `XMLDataSource` that only exposes `fetch_xml() -> str` (returns an XML string). We can't modify `XMLDataSource` (imagine it's from an external library).

We write an **Object Adapter** (`XMLToJSONAdapter`) that implements `JSONDataSource` and internally delegates to an `XMLDataSource` instance, translating XML into a dict.

In [ ]:
from abc import ABC, abstractmethod
import xml.etree.ElementTree as ET


# --- Target interface: what the client expects ---
class JSONDataSource(ABC):
    @abstractmethod
    def fetch_json(self) -> dict:
        """Return data as a plain dict."""
        raise NotImplementedError


In [ ]:
# --- Adaptee: existing/legacy class with an incompatible interface ---
class XMLDataSource:
    """Simulates a third-party class we cannot modify."""

    def fetch_xml(self) -> str:
        return "<user><id>42</id><name>Ada Lovelace</name></user>"


In [ ]:
# --- Adapter: implements Target, delegates to Adaptee, translates the call ---
class XMLToJSONAdapter(JSONDataSource):
    def __init__(self, xml_source: XMLDataSource):
        self._xml_source = xml_source

    def fetch_json(self) -> dict:
        xml_string = self._xml_source.fetch_xml()
        root = ET.fromstring(xml_string)
        return {child.tag: child.text for child in root}


In [ ]:
# --- Client: only knows about the JSONDataSource interface ---
def print_user_report(source: JSONDataSource) -> None:
    data = source.fetch_json()
    print(f"User #{data['id']}: {data['name']}")


legacy_source = XMLDataSource()
adapter = XMLToJSONAdapter(legacy_source)

print_user_report(adapter)  # Client never touches XML directly


**Key point**: `print_user_report` only depends on `JSONDataSource`. Swap in any adapter (or a native JSON source) without changing client code — that's the Open/Closed Principle at work.